# Getting predictions on New Data

In [71]:
import pandas as pd
data = input("input relative path to new data file (must have columns 'text' and 'label'): ")
data = pd.read_csv(data)
original_data = data.copy()

## Add required features

In [72]:
#libraries
import numpy as np
import re
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
df_train = pd.read_csv("train.csv")
data['text'] = data['text'].str.strip()
df_train['text'] = df_train['text'].str.strip()

from nltk.tokenize import word_tokenize
data['text'] = data['text'].apply(word_tokenize)
df_train['text'] = df_train['text'].apply(word_tokenize)


nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger_eng')
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet

lemmatizer = WordNetLemmatizer()

# Convert POS tags for WordNet
def get_wordnet_pos(tag):
    if tag.startswith('J'): 
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN   # default

def lemmatize_tokens(tokens):
    pos_tags = nltk.pos_tag(tokens)
    return [lemmatizer.lemmatize(word, get_wordnet_pos(tag)) 
            for word, tag in pos_tags]

data['text'] = data['text'].apply(lemmatize_tokens)
df_train['text'] = df_train['text'].apply(lemmatize_tokens)

import pandas as pd
import numpy as np
import string
import re
from collections import Counter
import ast

# print(data['text'])
data["tokens"] = data["text"]
#.apply(ast.literal_eval)
df_train["tokens"] = df_train["text"]

df_train_str = pd.read_csv("train.csv")

from sklearn.feature_extraction.text import TfidfVectorizer
import joblib
# vectorizer = joblib.load("tfidf_vectorizer.pkl")

vectorizer = TfidfVectorizer(lowercase=False)

train_tfidf = vectorizer.fit_transform(df_train_str['text'])
data_tfidf = vectorizer.transform(original_data['text'])




import ast
from nltk.sentiment import SentimentIntensityAnalyzer
import nltk

nltk.download('vader_lexicon', quiet = True)

sia = SentimentIntensityAnalyzer()

df_train['sentiment'] = df_train_str['text'].apply(lambda x: sia.polarity_scores(x)['compound'])
df_train['length_words'] = df_train_str['text'].apply(lambda x: len(x.split()))

data['sentiment'] = original_data['text'].apply(lambda x: sia.polarity_scores(x)['compound'])

data['length_words'] = original_data['text'].apply(lambda x: len(x.split()))


def extract_punctuation_features(text):
    features = {}
    features['exclamation_count'] = text.count('!')
    features['question_count'] = text.count('?')
    features['ellipsis_count'] = len(re.findall(r'\.{2,}', text))  # Two or more dots
    features['quote_count'] = text.count('"') + text.count("'")
    features['comma_count'] = text.count(',')
    features['period_count'] = text.count('.')
    features['semicolon_count'] = text.count(';')
    features['colon_count'] = text.count(':')
    features['dash_count'] = text.count('-') + text.count('—')
    features['multiple_exclamation'] = len(re.findall(r'!{2,}', text))
    features['multiple_question'] = len(re.findall(r'\?{2,}', text))
    features['mixed_punctuation'] = len(re.findall(r'(\?!|!\?)', text))
    total_punct = sum([features['exclamation_count'], features['question_count'],features['ellipsis_count'],features['quote_count'],features['semicolon_count'],features['colon_count'],
                       features['comma_count'], features['period_count'],features['dash_count']])
    text_length = len(text.split())
    features['punct_density'] = total_punct / max(text_length, 1)
    features['all_caps_words'] = len(re.findall(r'\b[A-Z]{2,}\b', text))
    features['punct_types_used'] = len(set(c for c in text if c in string.punctuation))
    
    return features

def create_punctuation_features(text_series):
    punct_features = text_series.apply(extract_punctuation_features) 
    return pd.DataFrame(punct_features.tolist())


data_punct = create_punctuation_features(original_data['text'])

data = pd.concat([data, data_punct], axis=1)

df_tain_punct = create_punctuation_features(df_train_str['text'])

df_train = pd.concat([df_train, df_tain_punct], axis=1)

def parse_token_list(text_str):
    try:
        return ast.literal_eval(text_str)
    except:
        return []
    
def get_first_last_words(df):
    tokens = df['text'].apply(parse_token_list)
    
    first_words = tokens.apply(lambda x: x[0] if len(x) > 0 else '<EMPTY>')
    last_words = tokens.apply(lambda x: x[-1] if len(x) > 0 else '<EMPTY>')
    
    return first_words, last_words

data_first, data_last = get_first_last_words(data)

df_train_first, df_train_last = get_first_last_words(df_train)

def build_position_vocabulary(words_series, top_n=200, min_freq=5):
    
    word_counts = Counter(words_series)
    
    filtered_words = {word: count for word, count in word_counts.items() 
                     if count >= min_freq}
    
    vocab = [word for word, _ in sorted(filtered_words.items(), 
                                       key=lambda x: x[1], 
                                       reverse=True)[:top_n]]
    
    return vocab

first_word_vocab = build_position_vocabulary(data_first, top_n=200, min_freq=5)
last_word_vocab = build_position_vocabulary(data_last, top_n=200, min_freq=5)

df_train_first_vocab = build_position_vocabulary(df_train_first, top_n=200, min_freq=5)
df_train_last_vocab = build_position_vocabulary(df_train_last, top_n=200, min_freq=5)

def create_position_features(first_words, last_words, first_vocab, last_vocab):
    
    features = {}
    
    for word in first_vocab:
        features[f'first_{word}'] = (first_words == word).astype(int)
    
    for word in last_vocab:
        features[f'last_{word}'] = (last_words == word).astype(int)
    
    return pd.DataFrame(features)

data_position = create_position_features(data_first, data_last, first_word_vocab, last_word_vocab)
# data = pd.concat([data, data_position], axis=1)

# data.to_csv('test_features.csv', index=False)
df_train_position = create_position_features(df_train_first, df_train_last, df_train_first_vocab, df_train_last_vocab)


from sklearn.feature_extraction.text import CountVectorizer

def identity_tokenizer(tokens):
    return tokens  

vectorizer = CountVectorizer(
    tokenizer=identity_tokenizer,
    preprocessor=lambda x: x, 
    ngram_range=(1,2),
    lowercase=False
)

import ast

# df_train = pd.read_csv('train.csv')
X_train_bow = vectorizer.fit_transform(df_train['text'])
X_data_bow = vectorizer.transform(original_data['text'])
word_count = np.asarray(X_data_bow.sum(axis=0)).flatten()
features = vectorizer.get_feature_names_out()
freq_list = list(zip(features, word_count))
freq_list.sort(key=lambda x: x[1], reverse=True)









import nltk
from nltk.corpus import wordnet
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger_eng')
def get_wordnet_pos(tag):
    if tag.startswith('J'): 
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN   # default

def tokens_to_pos_df(df, token_col='tokens'):
    words = []
    pos_tags = []

    for tokens in df[token_col]:
        tagged = nltk.pos_tag(tokens)   # list of (word, pos)
        for word, pos in tagged:
            words.append(word)
            pos_tags.append(pos)

    return pd.DataFrame({'word': words, 'pos': pos_tags})


pos_df = tokens_to_pos_df(df_train, 'text')

pos_data = tokens_to_pos_df(data, 'text')


pos_mapping = {
    # Nouns
    'NN': 'N', 'NNS': 'N', 'NNP': 'N', 'NNPS': 'N', 
    'PRP': 'N', 'PRP$': 'N', 'WP': 'N', 'WP$': 'N', 'EX': 'N',
    
    # Verbs
    'VB': 'V', 'VBD': 'V', 'VBG': 'V', 'VBN': 'V', 'VBP': 'V', 'VBZ': 'V',
    
    # Adjectives / Adverbs
    'JJ': 'A', 'JJR': 'A', 'JJS': 'A', 
    'RB': 'A', 'RBR': 'A', 'RBS': 'A', 'WRB': 'A',
    
    # Prepositions / Determiners / Modals / Conjunctions
    'IN': 'P', 'TO': 'P', 'DT': 'P', 'PDT': 'P', 'CC': 'P', 'MD': 'P',
    
    # Other
    'RP': 'O', 'POS': 'O', 'CD': 'O', 'UH': 'O', 'FW': 'O', 'SYM': 'O', '$': 'O', '#': 'O',
    
    # Punctuation
    '.': 'U', ',': 'U', ':': 'U'
}

# Convert to DataFrame
def words_with_pos(df, token_col='text'):
    """
    Takes a DataFrame with tokenized text and returns a DataFrame with:
    word | fine_pos | high_level_pos
    """
    rows = []
    for tokens in df[token_col]:
        pos_tags = nltk.pos_tag(tokens)
        for word, fine_pos in pos_tags:
            high_pos = pos_mapping.get(fine_pos, 'O')  # default 'O' if not mapped
            rows.append({'word': word, 'fine_pos': fine_pos, 'high_level_pos': high_pos})
    return pd.DataFrame(rows)

df_train_words = words_with_pos(df_train, token_col='text')
data_words = words_with_pos(data, token_col='text')

from nltk import pos_tag

nltk.download('averaged_perceptron_tagger_eng')

def get_high_level_pos(tag):
    if tag.startswith('N'):
        return 'N'
    elif tag.startswith('V'):
        return 'V'
    elif tag.startswith('J') or tag.startswith('R'):
        return 'A'
    elif tag in ['IN','TO','DT','PDT','CC','MD']:
        return 'P'
    elif tag in ['RP','POS','CD','UH','FW','SYM','$','#']:
        return 'O'
    elif tag in ['.',';',',',':']:
        return 'U'
    else:
        return 'O'

# Tag tokens in context and create new column
def tag_tokens(tokens):
    tagged = pos_tag(tokens)  # [('word','POS'), ...]
    high_level = [get_high_level_pos(tag) for _, tag in tagged]
    return high_level

data['pos_seq'] = data['text'].apply(tag_tokens)
df_train['pos_seq'] = df_train['text'].apply(tag_tokens)


def pos_counts(tokens):
    counts = {'N':0, 'V':0, 'A':0, 'P':0, 'O':0, 'U':0}
    for word, fine_pos in nltk.pos_tag(tokens):
        high_pos = pos_mapping.get(fine_pos, 'O') 
        counts[high_pos] += 1
    return counts

data['text'] = data['text'].apply(lambda x: x.split() if isinstance(x, str) else x)
df_train['text'] = df_train['text'].apply(lambda x: x.split() if isinstance(x, str) else x)

pos_features_data = data['text'].apply(pos_counts).apply(pd.Series)
pos_features_df_train = df_train['text'].apply(pos_counts).apply(pd.Series)
data = pd.concat([data, pos_features_data], axis=1)
df_train = pd.concat([df_train, pos_features_df_train], axis=1)


import ast
import pandas as pd

POS_TAGS = ["N", "V", "A", "P", "O", "U"]

# Create full bigram lexicon
BIGRAMS = [a + b for a in POS_TAGS for b in POS_TAGS]

def add_pos_bigrams(df):
    # Initialize all bigram columns
    for bg in BIGRAMS:
        df[bg] = 0

    # Fill counts
    for i, seq in df["pos_seq"].items():
        # Create bigrams from sequence
        seq_bigrams = [seq[j] + seq[j+1] for j in range(len(seq)-1)]

        # Count them
        counts = pd.Series(seq_bigrams).value_counts()

        # Assign to row
        for bg in BIGRAMS:
            df.at[i, bg] = counts.get(bg, 0)
    return df

data = add_pos_bigrams(data)
df_train = add_pos_bigrams(df_train)





[nltk_data] Downloading package punkt to
[nltk_data]     /Users/michelleshlivko/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/michelleshlivko/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/michelleshlivko/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/michelleshlivko/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
/opt/miniconda3/lib/python3.13/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/michelleshlivko/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[n

## BiLSTM

In [74]:
import torch
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torch.nn as nn
import torch

class model2_small_more_nodes(nn.Module):
    def __init__(self, vocab_size, pos_vocab_size, 
                 lstm_hidden=32, 
                 tfidf_dim=100, 
                 punct_dim=13,
                 use_tfidf=True, 
                 use_sentiment=True, 
                 use_punct=True, 
                 num_hidden=64,
                 output_size=2):
        super().__init__()

        self.use_tfidf = use_tfidf
        self.use_sentiment = use_sentiment
        self.use_punct = use_punct

        self.word_emb = nn.Embedding(vocab_size, 128, padding_idx=0)
        self.pos_emb  = nn.Embedding(pos_vocab_size, 32, padding_idx=0)

        self.lstm = nn.LSTM(128+32, lstm_hidden, batch_first=True, bidirectional=True)

        aux_size = 0
        if use_tfidf:
            aux_size += tfidf_dim
        if use_sentiment:
            aux_size += 1
        if use_punct:
            aux_size += punct_dim

        self.fc = nn.Sequential(
            nn.Linear(lstm_hidden*2 + aux_size, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Linear(32, output_size)
        )

    def forward(self, word_ids, pos_ids, aux_features=None):
        w = self.word_emb(word_ids)
        p = self.pos_emb(pos_ids)
        x = torch.cat([w, p], dim=-1)

        out, _ = self.lstm(x)
        h = out.size(2)//2
        lstm_feat = torch.cat([out[:, -1, :h], out[:, 0, h:]], dim=1)

        if aux_features is not None:
            combined = torch.cat([lstm_feat, aux_features], dim=1)
        else:
            combined = lstm_feat

        return self.fc(combined)




def load_checkpoint(
    path,
    model_cls,
    optimizer_cls=None,
    scheduler_cls=None,
    device="cpu"
):
    checkpoint = torch.load(path, map_location=device)

    config = checkpoint["config"]
    model = model_cls(**config["model_args"])
    model.load_state_dict(checkpoint["model_state"])
    model.to(device)
    model.eval()

    optimizer = None
    if optimizer_cls is not None:
        optimizer = optimizer_cls(model.parameters(), lr=config["lr"])
        optimizer.load_state_dict(checkpoint["optimizer_state"])

    scheduler = None
    if scheduler_cls is not None and checkpoint["scheduler_state"] is not None:
        scheduler = scheduler_cls(optimizer)
        scheduler.load_state_dict(checkpoint["scheduler_state"])

    return {
        "model": model,
        "optimizer": optimizer,
        "scheduler": scheduler,
        "word_vocab": checkpoint["word_vocab"],
        "pos_vocab": checkpoint["pos_vocab"],
        "config": config
    }
    
checkpoint = load_checkpoint(
    "bilstm_model.pt",
    model_cls=model2_small_more_nodes,
    optimizer_cls=torch.optim.SGD, 
    scheduler_cls=lambda optim: torch.optim.lr_scheduler.StepLR(optim, step_size=5, gamma=0.1),
    device="cpu"
)

bilstm_model = checkpoint["model"]
optimizer = checkpoint["optimizer"]
scheduler = checkpoint["scheduler"]
word_vocab = checkpoint["word_vocab"]
pos_vocab  = checkpoint["pos_vocab"]

bilstm_model.eval()



# import pandas as pd
# df_train = pd.read_csv("train_features.csv")
# df_valid = pd.read_csv("valid_features.csv")
# df_test = pd.read_csv("test_features.csv")


import numpy as np
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
from collections import Counter
import ast

def bilstm_processing(df_train, other_dfs, tfidf_data, svd_components=100):
    """
    Preprocess LSTM and TF-IDF data for stacking.
    
    Parameters:
    df_train : must contain features
    other_dfs : must contain features
        List of additional datasets (validation/test) to preprocess.
    tfidf_data : Paths to npz TF-IDF files corresponding to df_train + other_dfs.
 
    """
    assert len(other_dfs) + 1 == len(tfidf_data), "Must provide tfidf files for all datasets including train"
    
    # --- Load and transform TF-IDF ---
    # def load_npz(path):
    #     npz = np.load(path)
    #     return csr_matrix((npz["data"], npz["indices"], npz["indptr"]), shape=tuple(npz["shape"]))
    
    def load_npz(path_or_matrix):
        if isinstance(path_or_matrix, csr_matrix):
            return path_or_matrix
        
        npz = np.load(path_or_matrix)
        return csr_matrix((npz["data"], npz["indices"], npz["indptr"]), shape=tuple(npz["shape"]))

    
    X_tfidf_train = load_npz(tfidf_data[0])
    svd = TruncatedSVD(n_components=svd_components, random_state=42)
    X_svd_train = svd.fit_transform(X_tfidf_train)

    
    
    # --- Process text columns ---
    # for df in [df_train] + other_dfs:
        # df["text"] = df["text"].apply(ast.literal_eval)
        # df["pos_seq"] = df["pos_seq"].apply(ast.literal_eval)
    
    # --- Build vocabularies from training data ---
    def build_vocab(tokens_list, min_freq=1):
        counter = Counter()
        for sentence in tokens_list:
            for token in sentence:
                counter.update([token])
        vocab = {"<pad>": 0, "<unk>": 1}
        for tok, freq in counter.items():
            if freq >= min_freq:
                vocab[tok] = len(vocab)
        return vocab
    
    word_vocab = build_vocab(df_train["text"])
    pos_vocab = build_vocab(df_train["pos_seq"])
    
    # --- Encode sequences ---
    def encode(tokens, vocab):
        return [vocab.get(tok, vocab["<unk>"]) for tok in tokens]
    
    for df in [df_train] + other_dfs:
        df["word_ids"] = df["text"].apply(lambda x: encode(x, word_vocab))
        df["pos_ids"]  = df["pos_seq"].apply(lambda x: encode(x, pos_vocab))
    
    # --- Pad sequences ---
    max_len = max(len(seq) for seq in df_train["text"])
    
    def pad(seq, length, pad_value=0):
        return seq + [pad_value] * (length - len(seq))
    
    for df in [df_train] + other_dfs:
        df["word_padded"] = df["word_ids"].apply(lambda x: pad(x, max_len))
        df["pos_padded"]  = df["pos_ids"].apply(lambda x: pad(x, max_len))
    
    # --- TF-IDF SVD features ---
    X_svd_others = []
    for i, df in enumerate(other_dfs, start=1):
        X_tfidf_other = load_npz(tfidf_data[i])
        X_svd_other = svd.transform(X_tfidf_other)
        X_svd_others.append(X_svd_other)
    
    # --- Assign SVD features to DataFrames ---
    df_train["tfidf_svd"] = list(X_svd_train)
    df_train["tfidf_order"] = [X_svd_train[i] for i in range(X_svd_train.shape[0])]

    for df, X_svd_other in zip(other_dfs, X_svd_others):
        df["tfidf_svd"] = list(X_svd_other)
        df["tfidf_order"] = [X_svd_other[i] for i in range(X_svd_other.shape[0])]

        
    
    # --- Output dictionary ---
    processed = {
        "word_vocab": word_vocab,
        "pos_vocab": pos_vocab,
        "train": df_train,
        #     {
        #     "word_padded": np.stack(df_train["word_padded"].values),
        #     "pos_padded": np.stack(df_train["pos_padded"].values),
        #     "tfidf_svd": np.stack(df_train["tfidf_svd"].values),
        #     "labels": df_train["label"].values
        # },
        "others": other_dfs,

            #[
        #     {
        #         "word_padded": np.stack(df["word_padded"].values),
        #         "pos_padded": np.stack(df["pos_padded"].values),
        #         "tfidf_svd": np.stack(df["tfidf_svd"].values),
        #         "labels": df["label"].values
        #     }
        #     for df in other_dfs
        # ],
        "svd": svd
    }
    
    return processed

processed = bilstm_processing(
    df_train=df_train,
    other_dfs=[data],
    tfidf_data=[train_tfidf, data_tfidf]
)

train_data = processed["train"]
data_processed = processed["others"][0]
# test_data  = processed["others"][1]

# word_vocab = processed['word_vocab']
# pos_vocab = processed['pos_vocab']


from torch.utils.data import Dataset, DataLoader
import torch.nn.utils.rnn as rnn_utils

from torch.utils.data import Dataset
import torch

class TextPosDataset(Dataset):
    def __init__(self, df, use_tfidf=True, use_sentiment=True, use_punct=True):
        self.word_ids = df["word_ids"].tolist()
        self.pos_ids = df["pos_ids"].tolist()
        self.label = df["label"].tolist()

        self.use_tfidf = use_tfidf
        self.use_sentiment = use_sentiment
        self.use_punct = use_punct

        if use_tfidf:
            self.tfidf = df["tfidf_order"].tolist()
        if use_sentiment:
            self.sentiment = df["sentiment"].tolist()
        if use_punct:
            self.punct_features = df[[
                'exclamation_count','question_count','ellipsis_count','quote_count',
                'comma_count','period_count','semicolon_count','colon_count','dash_count',
                'multiple_exclamation','multiple_question','mixed_punctuation','punct_density'
            ]].values.tolist()

    def __len__(self):
        return len(self.label)

    def __getitem__(self, idx):
        items = [
            torch.tensor(self.word_ids[idx]),
            torch.tensor(self.pos_ids[idx])
        ]

        # optional auxiliary features
        aux_list = []
        if self.use_tfidf:
            aux_list.append(torch.tensor(self.tfidf[idx], dtype=torch.float))
        if self.use_sentiment:
            aux_list.append(torch.tensor([self.sentiment[idx]], dtype=torch.float))
        if self.use_punct:
            aux_list.append(torch.tensor(self.punct_features[idx], dtype=torch.float))

        # concatenate all auxiliary features
        if aux_list:
            aux_features = torch.cat(aux_list)
        else:
            aux_features = torch.tensor([])  # empty tensor if none selected

        items.append(aux_features)
        items.append(torch.tensor(self.label[idx]))
        return tuple(items)

def collate(batch):
    words, pos, aux_features, labels = zip(*batch)
    
    # pad sequences
    words = rnn_utils.pad_sequence(words, batch_first=True, padding_value=0)
    pos   = rnn_utils.pad_sequence(pos, batch_first=True, padding_value=0)
    
    if aux_features[0].numel() > 0:
        aux_features = torch.stack(aux_features)
    else:
        aux_features = None  # no aux features used

    labels = torch.stack(labels)
    return words, pos, aux_features, labels


valid_dataset = TextPosDataset(data)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False, collate_fn=collate)

# test_dataset = TextPosDataset(df_test)
# test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate)



import torch

def get_lstm_predictions(model, data_loader, device="cpu"):
    model.eval()
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for words, pos, aux, labels in data_loader:
            words, pos, aux = words.long().to(device), pos.long().to(device), aux.float().to(device)
            outputs = model(words, pos, aux)  # shape [batch_size, num_classes]
            probs = torch.softmax(outputs, dim=1)  # convert logits to probabilities
            all_probs.append(probs.cpu().numpy())
            all_labels.append(labels.numpy())

    all_probs = np.vstack(all_probs)  # shape [num_samples, num_classes]
    all_labels = np.concatenate(all_labels)  # shape [num_samples]
    return all_probs, all_labels

val_probs, val_labels = get_lstm_predictions(bilstm_model, valid_loader, device="cpu")

validation_predictions = pd.DataFrame({
    "prediction": np.argmax(val_probs, axis=1),
    "probability": np.max(val_probs, axis=1),
    "label": val_labels
})
validation_predictions.to_csv("bilstm_Newdata_predictions.csv", index=False)



from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import numpy as np

print("Validation Accuracy:", accuracy_score(val_labels, np.argmax(val_probs, axis=1)))
print("Validation Confusion Matrix:\n", confusion_matrix(val_labels, np.argmax(val_probs, axis=1)))
print("Validation Classification Report:\n", classification_report(val_labels, np.argmax(val_probs, axis=1)))


Validation Accuracy: 0.7028985507246377
Validation Confusion Matrix:
 [[396 130]
 [157 283]]
Validation Classification Report:
               precision    recall  f1-score   support

           0       0.72      0.75      0.73       526
           1       0.69      0.64      0.66       440

    accuracy                           0.70       966
   macro avg       0.70      0.70      0.70       966
weighted avg       0.70      0.70      0.70       966



In [ ]:
# import numpy as np
# from scipy.sparse import csr_matrix
# from sklearn.decomposition import TruncatedSVD
# from collections import Counter
# import ast
# import torch
# from torch.utils.data import Dataset, DataLoader
# import torch.nn.utils.rnn as rnn_utils

# # --- Preprocessing function ---
# def bilstm_processing(df_train, other_dfs, tfidf_data, svd_components=100):
#     """
#     Preprocess LSTM and TF-IDF data for stacking.
    
#     df_train: training DataFrame
#     other_dfs: list of validation/test DataFrames
#     tfidf_data: list of npz paths or csr_matrix objects corresponding to df_train + other_dfs
#     """
    
#     assert len(other_dfs) + 1 == len(tfidf_data), "Must provide TF-IDF for all datasets including train"
    
#     # --- Load TF-IDF ---
#     def load_npz(path_or_matrix):
#         if isinstance(path_or_matrix, csr_matrix):
#             return path_or_matrix
#         npz = np.load(path_or_matrix)
#         return csr_matrix((npz["data"], npz["indices"], npz["indptr"]), shape=tuple(npz["shape"]))
    
#     X_tfidf_train = load_npz(tfidf_data[0])
#     svd = TruncatedSVD(n_components=svd_components, random_state=42)
#     X_svd_train = svd.fit_transform(X_tfidf_train)
    
#     # --- Convert text and POS columns from strings to lists ---
#     # for df in [df_train] + other_dfs:
#     #     df["text"] = df["text"].apply(ast.literal_eval)
#     #     df["pos_seq"] = df["pos_seq"].apply(ast.literal_eval)
    
#     # --- Build vocabularies from training data ---
#     def build_vocab(tokens_list, min_freq=1):
#         counter = Counter()
#         for sentence in tokens_list:
#             counter.update(sentence)
#         vocab = {"<pad>": 0, "<unk>": 1}
#         for tok, freq in counter.items():
#             if freq >= min_freq:
#                 vocab[tok] = len(vocab)
#         return vocab

#     word_vocab = build_vocab(df_train["text"])
#     pos_vocab = build_vocab(df_train["pos_seq"])
    
#     # --- Encode sequences ---
#     def encode(tokens, vocab):
#         return [vocab.get(tok, vocab["<unk>"]) for tok in tokens]
    
#     for df in [df_train] + other_dfs:
#         df["word_ids"] = df["text"].apply(lambda x: encode(x, word_vocab))
#         df["pos_ids"]  = df["pos_seq"].apply(lambda x: encode(x, pos_vocab))
    
#     # --- Pad sequences using max_len from training ---
#     max_len = max(len(seq) for seq in df_train["text"])
#     def pad(seq, length, pad_value=0):
#         return seq + [pad_value] * (length - len(seq))
    
#     for df in [df_train] + other_dfs:
#         df["word_padded"] = df["word_ids"].apply(lambda x: pad(x, max_len))
#         df["pos_padded"]  = df["pos_ids"].apply(lambda x: pad(x, max_len))
    
#     # --- Transform TF-IDF SVD for other datasets ---
#     X_svd_others = []
#     for i, df in enumerate(other_dfs, start=1):
#         X_tfidf_other = load_npz(tfidf_data[i])
#         X_svd_other = svd.transform(X_tfidf_other)
#         X_svd_others.append(X_svd_other)
    
#     # --- Assign SVD features to DataFrames ---
#     df_train["tfidf_order"] = [X_svd_train[i] for i in range(X_svd_train.shape[0])]
#     for df, X_svd_other in zip(other_dfs, X_svd_others):
#         df["tfidf_order"] = [X_svd_other[i] for i in range(X_svd_other.shape[0])]
    
#     # --- Return processed data ---
#     processed = {
#         "word_vocab": word_vocab,
#         "pos_vocab": pos_vocab,
#         "train": df_train,
#         "others": other_dfs,
#         "svd": svd
#     }
    
#     return processed


# # --- Dataset class ---
# class TextPosDataset(Dataset):
#     def __init__(self, df, use_tfidf=True, use_sentiment=True, use_punct=True):
#         self.word_ids = df["word_padded"].tolist()
#         self.pos_ids = df["pos_padded"].tolist()
#         self.label = df["label"].tolist()

#         self.use_tfidf = use_tfidf
#         self.use_sentiment = use_sentiment
#         self.use_punct = use_punct

#         if use_tfidf:
#             self.tfidf = df["tfidf_order"].tolist()
#         if use_sentiment and "sentiment" in df.columns:
#             self.sentiment = df["sentiment"].tolist()
#         if use_punct:
#             self.punct_features = df[[
#                 'exclamation_count','question_count','ellipsis_count','quote_count',
#                 'comma_count','period_count','semicolon_count','colon_count','dash_count',
#                 'multiple_exclamation','multiple_question','mixed_punctuation','punct_density'
#             ]].values.tolist() if all(col in df.columns for col in [
#                 'exclamation_count','question_count','ellipsis_count','quote_count',
#                 'comma_count','period_count','semicolon_count','colon_count','dash_count',
#                 'multiple_exclamation','multiple_question','mixed_punctuation','punct_density'
#             ]) else [[0]*13]*len(df)

#     def __len__(self):
#         return len(self.label)

#     def __getitem__(self, idx):
#         items = [
#             torch.tensor(self.word_ids[idx], dtype=torch.long),
#             torch.tensor(self.pos_ids[idx], dtype=torch.long)
#         ]

#         aux_list = []
#         if self.use_tfidf:
#             aux_list.append(torch.tensor(self.tfidf[idx], dtype=torch.float))
#         if self.use_sentiment:
#             aux_list.append(torch.tensor([self.sentiment[idx]], dtype=torch.float))
#         if self.use_punct:
#             aux_list.append(torch.tensor(self.punct_features[idx], dtype=torch.float))

#         aux_features = torch.cat(aux_list) if aux_list else torch.tensor([], dtype=torch.float)
#         items.append(aux_features)
#         items.append(torch.tensor(self.label[idx], dtype=torch.long))
#         return tuple(items)


# def collate(batch):
#     words, pos, aux_features, labels = zip(*batch)
    
#     words = rnn_utils.pad_sequence(words, batch_first=True, padding_value=0)
#     pos   = rnn_utils.pad_sequence(pos, batch_first=True, padding_value=0)
    
#     if aux_features[0].numel() > 0:
#         aux_features = torch.stack(aux_features)
#     else:
#         aux_features = None

#     labels = torch.stack(labels)
#     return words, pos, aux_features, labels



# processed = bilstm_processing(
#     df_train=df_train,
#     other_dfs=[data],
#     tfidf_data=[train_tfidf, data_tfidf]  # can be .npz paths or csr_matrix
# )

# train_data = processed["train"]
# valid_data = processed["others"][0]

# valid_dataset = TextPosDataset(valid_data)
# valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False, collate_fn=collate)


# import torch

# def get_lstm_predictions(model, data_loader, device="cpu"):
#     model.eval()
#     all_probs = []
#     all_labels = []

#     with torch.no_grad():
#         for words, pos, aux, labels in data_loader:
#             words, pos, aux = words.long().to(device), pos.long().to(device), aux.float().to(device)
#             outputs = model(words, pos, aux)  # shape [batch_size, num_classes]
#             probs = torch.softmax(outputs, dim=1)  # convert logits to probabilities
#             all_probs.append(probs.cpu().numpy())
#             all_labels.append(labels.numpy())

#     all_probs = np.vstack(all_probs)  # shape [num_samples, num_classes]
#     all_labels = np.concatenate(all_labels)  # shape [num_samples]
#     return all_probs, all_labels

# val_probs, val_labels = get_lstm_predictions(bilstm_model, valid_loader, device="cpu")

# validation_predictions = pd.DataFrame({
#     "prediction": np.argmax(val_probs, axis=1),
#     "probability": np.max(val_probs, axis=1),
#     "label": val_labels
# })
# validation_predictions.to_csv("bilstm_Newdata_predictions.csv", index=False)



# from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
# import numpy as np

# print("Validation Accuracy:", accuracy_score(val_labels, np.argmax(val_probs, axis=1)))
# print("Validation Confusion Matrix:\n", confusion_matrix(val_labels, np.argmax(val_probs, axis=1)))
# print("Validation Classification Report:\n", classification_report(val_labels, np.argmax(val_probs, axis=1)))


Validation Accuracy: 0.6977225672877847
Validation Confusion Matrix:
 [[407 119]
 [173 267]]
Validation Classification Report:
               precision    recall  f1-score   support

           0       0.70      0.77      0.74       526
           1       0.69      0.61      0.65       440

    accuracy                           0.70       966
   macro avg       0.70      0.69      0.69       966
weighted avg       0.70      0.70      0.70       966



In [ ]:
# logistic regression
